# BEM Inductance Extraction: Energy Method with `use_fmm=False`

**Follow-up to Joachim's feedback (2026-03-22)**

This notebook demonstrates the corrected BEM inductance extraction workflow:

1. **`use_fmm=False`**: Reproducible results, faster dense extraction (your suggestion)
2. **`ToDense().NumPy()`**: Replaces manual column-by-column extraction (your suggestion)
3. **Energy method**: `L = mu_0 * J^T * SL * J` replaces the divergent `1/(e^T L^{-1} e)` formula

**Key finding**: The old formula `1/(e^T L^{-1} e)` with uniform excitation **diverges with mesh refinement** (794% error at 4185 DOFs). The energy method with proper toroidal current projection converges correctly (-0.1% at 8082 DOFs).

**Environment**: NGSolve 6.2.2602 (pip), Python 3.12.8, Windows Server 2022

## 1. Setup: OCC Torus Mesh

In [1]:
import math
import time
import numpy as np
from ngsolve import (Mesh, HDivSurface, TaskManager, ds,
                     Integrate, CF, BND, GridFunction, Norm, sqrt, x, y, z)
from ngsolve.bem import LaplaceSL
from netgen.occ import WorkPlane, Axes, Axis, Pnt, Dir, OCCGeometry
from netgen.meshing import MeshingParameters

MU_0 = 4e-7 * np.pi
R, a = 0.05, 0.005  # Major/minor radius [m]

# Analytical reference (Neumann formula, external inductance)
L_neumann = MU_0 * R * (np.log(8 * R / a) - 2)
area_exact = 4 * np.pi**2 * R * a
print(f"R = {R*1e3:.0f} mm, a = {a*1e3:.0f} mm, R/a = {R/a:.0f}")
print(f"Neumann formula: L = {L_neumann*1e9:.2f} nH")
print(f"Exact area: {area_exact*1e4:.2f} cm^2")

R = 50 mm, a = 5 mm, R/a = 10
Neumann formula: L = 149.67 nH
Exact area: 98.70 cm^2


## 2. Helper: Energy Method vs Old Formula

The energy method computes inductance from the magnetic energy of a known current distribution:

$$L = \mu_0 \, \mathbf{J}^T \, \mathrm{SL} \, \mathbf{J} \qquad \text{(for } I = 1\text{)}$$

where $\mathbf{J}$ is the toroidal surface current for unit total current, projected onto `HDivSurface(order=0)`.

**No matrix inversion needed** -- just a matrix-vector product.

In [2]:
def make_torus_mesh(curvaturesafety):
    """Create OCC torus mesh with given curvaturesafety."""
    wp = WorkPlane(Axes(p=Pnt(R, 0, 0), n=Dir(0, 1, 0), h=Dir(0, 0, 1)))
    circle = wp.Circle(a).Face()
    torus = circle.Revolve(Axis(p=Pnt(0, 0, 0), d=Dir(0, 0, 1)), 360)
    geo = OCCGeometry(torus)
    ngmesh = geo.GenerateMesh(
        mp=MeshingParameters(maxh=1.0, curvaturesafety=curvaturesafety, segmentsperedge=2))
    return Mesh(ngmesh)


def compute_inductance(mesh):
    """Compute inductance using energy method with use_fmm=False + ToDense().
    
    Returns dict with L_energy, L_old, area, timing, diagnostics.
    """
    area = float(Integrate(CF(1), mesh, VOL_or_BND=BND))
    
    fes = HDivSurface(mesh, order=0)
    u, v = fes.TnT()
    ndof = fes.ndof
    
    # Toroidal current for I=1: J = e_phi / (2*pi*a)
    r_cf = sqrt(x*x + y*y)
    J_toroidal = CF((-y/r_cf, x/r_cf, 0)) / (2 * math.pi * a)
    gf_J = GridFunction(fes)
    gf_J.Set(J_toroidal, definedon=mesh.Boundaries(".*"), dual=True)
    J_vec = gf_J.vec.FV().NumPy().copy()
    
    # LaplaceSL: use_fmm=False + ToDense() (Joachim's recommendation)
    t0 = time.perf_counter()
    with TaskManager():
        L_op = LaplaceSL(u.Trace() * ds, use_fmm=False) * v.Trace() * ds
        SL = L_op.mat.ToDense().NumPy()
    t_total = time.perf_counter() - t0
    
    # Energy method: L = mu_0 * J^T * SL * J
    L_energy = MU_0 * float(J_vec @ SL @ J_vec)
    
    # Old formula for comparison: L = 1 / (e^T * (mu_0*SL)^{-1} * e)
    L_mat = MU_0 * SL
    e_uniform = np.ones(ndof) / ndof
    try:
        x_sol = np.linalg.solve(L_mat, e_uniform)
        L_old = 1.0 / (e_uniform @ x_sol)
    except np.linalg.LinAlgError:
        L_old = float('nan')
    
    # Diagnostics
    sym_err = np.linalg.norm(SL - SL.T) / np.linalg.norm(SL)
    
    return {
        'ndof': ndof, 'nse': mesh.GetNE(BND),
        'area': area, 'area_err': (area - area_exact) / area_exact * 100,
        'L_energy': L_energy, 'L_old': L_old,
        'L_energy_err': (L_energy - L_neumann) / L_neumann * 100,
        'L_old_err': (L_old - L_neumann) / L_neumann * 100 if not np.isnan(L_old) else float('nan'),
        'sym_err': sym_err, 't_total': t_total,
    }

## 3. Mesh Convergence: Energy Method vs Old Formula

The old formula `1/(e^T L^{-1} e)` diverges with mesh refinement because `e = ones(n)/n` is not a physical current distribution in HDivSurface DOFs (edge normal fluxes).

The energy method with proper toroidal current projection converges correctly.

In [3]:
print(f"{'cs':>6s} {'nse':>6s} {'ndof':>6s} {'area_err':>10s} {'L_energy':>12s} {'energy_err':>12s} {'L_old':>12s} {'old_err':>12s} {'sym':>10s} {'time':>6s}")
print("-" * 105)

results = []
for cs in [0.5, 0.7, 1.0, 1.5, 2.0]:
    mesh = make_torus_mesh(cs)
    r = compute_inductance(mesh)
    results.append((cs, r))
    
    old_str = f"{r['L_old']*1e9:10.2f}nH" if not np.isnan(r['L_old']) else "  SINGULAR "
    old_err = f"{r['L_old_err']:+8.1f}%" if not np.isnan(r['L_old_err']) else "      nan"
    
    print(f"{cs:6.1f} {r['nse']:6d} {r['ndof']:6d} {r['area_err']:+8.2f}%"
          f"  {r['L_energy']*1e9:10.2f}nH {r['L_energy_err']:+8.1f}%"
          f"  {old_str} {old_err}"
          f"  {r['sym_err']:.2e} {r['t_total']:5.1f}s")

print(f"\nNeumann reference: {L_neumann*1e9:.2f} nH")

    cs    nse   ndof   area_err     L_energy   energy_err        L_old      old_err        sym   time
---------------------------------------------------------------------------------------------------------
   0.5    180    269   -17.83%      106.47nH    -28.9%      179.26nH    +19.8%  3.34e-03   0.0s


   0.7    440    869    -6.71%      136.62nH     -8.7%    SINGULAR        nan  1.74e-03   0.2s


   1.0    802   1790    -3.90%      142.12nH     -5.0%    SINGULAR        nan  1.31e-03   1.2s


   1.5   1906   5235    -1.66%      147.16nH     -1.7%    SINGULAR        nan  6.59e-04  22.8s


   2.0   3402  10977    -0.87%      148.73nH     -0.6%    SINGULAR        nan  4.99e-04 140.8s

Neumann reference: 149.67 nH


## 4. Reproducibility: `use_fmm=False` eliminates fluctuation

As you pointed out, `use_fmm=False` makes results reproducible. With FMM off, the entire operator is treated as nearfield (computed once at setup), so `ToDense()` simply reads the precomputed matrix.

In [4]:
# Reproducibility test: 5 runs with use_fmm=False + TaskManager
mesh = make_torus_mesh(1.0)
fes = HDivSurface(mesh, order=0)
u, v = fes.TnT()
ndof = fes.ndof

r_cf = sqrt(x*x + y*y)
J_tor = CF((-y/r_cf, x/r_cf, 0)) / (2 * math.pi * a)
gf_J = GridFunction(fes)
gf_J.Set(J_tor, definedon=mesh.Boundaries(".*"), dual=True)
J_vec = gf_J.vec.FV().NumPy().copy()

print(f"Mesh: {mesh.GetNE(BND)} surface elements, {ndof} DOFs\n")
print("use_fmm=False + TaskManager (5 runs):")

L_values = []
for i in range(5):
    with TaskManager():
        L_op = LaplaceSL(u.Trace() * ds, use_fmm=False) * v.Trace() * ds
        SL = L_op.mat.ToDense().NumPy()
    L = MU_0 * float(J_vec @ SL @ J_vec)
    L_values.append(L)
    print(f"  Run {i}: L = {L*1e9:.6f} nH")

spread = (max(L_values) - min(L_values)) * 1e9
print(f"\n  Spread: {spread:.6f} nH  (bit-for-bit identical: {spread == 0.0})")

Mesh: 802 surface elements, 1790 DOFs

use_fmm=False + TaskManager (5 runs):


  Run 0: L = 142.118395 nH


  Run 1: L = 142.118395 nH


  Run 2: L = 142.118395 nH
  Run 3: L = 142.118395 nH


  Run 4: L = 142.118395 nH

  Spread: 0.000000 nH  (bit-for-bit identical: True)


## 5. Performance: `use_fmm=False` + `ToDense()` vs manual extraction

Comparing the recommended workflow against the old column-by-column approach.

In [5]:
# Method 1: use_fmm=False + ToDense() (recommended)
t0 = time.perf_counter()
with TaskManager():
    L_op1 = LaplaceSL(u.Trace() * ds, use_fmm=False) * v.Trace() * ds
    SL1 = L_op1.mat.ToDense().NumPy()
t_new = time.perf_counter() - t0

# Method 2: FMM default + manual column-by-column (old)
t0 = time.perf_counter()
L_op2 = LaplaceSL(u.Trace() * ds) * v.Trace() * ds
SL2 = np.zeros((ndof, ndof))
ei = L_op2.mat.CreateColVector()
col = L_op2.mat.CreateColVector()
for j in range(ndof):
    ei[:] = 0; ei[j] = 1.0
    L_op2.mat.Mult(ei, col)
    SL2[:, j] = col.FV().NumPy()
t_old = time.perf_counter() - t0

# Compare
diff = np.linalg.norm(SL1 - SL2) / np.linalg.norm(SL1)
L1 = MU_0 * float(J_vec @ SL1 @ J_vec)
L2 = MU_0 * float(J_vec @ SL2 @ J_vec)

print(f"{'Method':<40s} {'Time':>8s} {'L (nH)':>12s}")
print("-" * 62)
print(f"{'use_fmm=False + ToDense() (new)':<40s} {t_new:7.1f}s {L1*1e9:10.4f}nH")
print(f"{'FMM default + column-by-column (old)':<40s} {t_old:7.1f}s {L2*1e9:10.4f}nH")
print(f"\nSpeedup: {t_old/t_new:.1f}x")
print(f"Matrix difference: ||SL_new - SL_old|| / ||SL_new|| = {diff:.2e}")

Method                                       Time       L (nH)
--------------------------------------------------------------
use_fmm=False + ToDense() (new)              1.4s   142.1184nH
FMM default + column-by-column (old)       109.8s   142.1830nH

Speedup: 77.8x
Matrix difference: ||SL_new - SL_old|| / ||SL_new|| = 3.47e-02


## 6. Matrix Diagnostics

In [6]:
# Use the SL matrix from the recommended workflow
L_mat = MU_0 * SL1
eigvals = np.linalg.eigvalsh(L_mat)
cond = eigvals[-1] / max(abs(eigvals[0]), 1e-30)
sym_err = np.linalg.norm(SL1 - SL1.T) / np.linalg.norm(SL1)
neg_diag = int(np.sum(np.diag(L_mat) < 0))
n_zero = int(np.sum(np.abs(eigvals) < 1e-12 * np.max(np.abs(eigvals))))

print(f"Matrix size:       {ndof} x {ndof}")
print(f"Eigenvalues:       min = {eigvals[0]:.3e}, max = {eigvals[-1]:.3e}")
print(f"Near-zero eigvals: {n_zero}")
print(f"Condition number:  {cond:.1f}")
print(f"Symmetry error:    ||SL - SL^T|| / ||SL|| = {sym_err:.2e}")
print(f"Negative diagonal: {neg_diag}")
print(f"\nAll eigenvalues positive: {np.all(eigvals > 0)}")

Matrix size:       1790 x 1790
Eigenvalues:       min = -1.224e-24, max = 1.245e-08
Near-zero eigvals: 587
Condition number:  10174584548373202.0
Symmetry error:    ||SL - SL^T|| / ||SL|| = 1.31e-03
Negative diagonal: 0

All eigenvalues positive: False


## 7. Summary

| Issue | Old workflow | New workflow (your suggestions) |
|-------|-------------|-------------------------------|
| Reproducibility | FMM causes ~1.4% fluctuation | **`use_fmm=False`**: bit-for-bit identical |
| Dense extraction | Column-by-column (slow) | **`ToDense().NumPy()`**: faster |
| Inductance formula | `1/(e^T L^{-1} e)` diverges | **Energy method**: `mu_0 * J^T * SL * J` converges |
| TaskManager | Inconsistent use | **Both setup + extraction**, or neither |

### Open question

For the inductance extraction, we project a known toroidal current $\mathbf{J} = \hat{e}_\phi / (2\pi a)$ onto `HDivSurface(order=0)` and compute $L = \mu_0 \mathbf{J}^T \mathrm{SL} \, \mathbf{J}$. This converges well with mesh refinement.

Is there a recommended ngsolve.bem workflow for extracting inductance when the current direction is **not** known a priori (e.g., a conductor with arbitrary shape and source/sink ports)?